In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
# CKP = "../outputs/20250128_163111"
CKP = "/restricteddata/ukaea/checkpoints/correct_ifft_400M/"
device = "cuda"

In [ ]:
import yaml
from omegaconf import OmegaConf

from utils import load_model_and_config
from models import get_model

from dataset.cyclone import CycloneDataset

cfg = OmegaConf.create(yaml.safe_load(open(f"{CKP}/config.yaml", "r")))

# traindata = CycloneDataset(
#     active_keys=cfg.dataset.active_keys,
#     path=cfg.dataset.path,
#     split="train",
#     random_seed=cfg.seed,
#     test_ratio=0.0,
#     normalization=cfg.dataset.normalization,
#     spatial_ifft=cfg.dataset.spatial_ifft,
#     in_memory=False,
#     input_seq_length=cfg.model.input_seq_length,
#     target_seq_length=cfg.model.bundle_seq_length,
#     trajectories=cfg.dataset.training_trajectories,
# )

data = CycloneDataset(
    active_keys=cfg.dataset.active_keys,
    split="val",
    random_seed=cfg.seed,
    normalization=None,
    spatial_ifft=False,
    in_memory=False,
    bundle_seq_length=cfg.model.bundle_seq_length,
    trajectories=["cyclone4_2_2.h5"],
)

print(f"Val: {len(data)}")

In [7]:
cfg.model.swin.abs_pe = False

In [ ]:
model = get_model(cfg, dataset=data)

model, _, _ = load_model_and_config(f"{CKP}/best.pth", model, device)

model = model.to(device)
model.eval()

In [76]:
import torch
import numpy as np

from utils import expand_as


def do_ifft(x):
    x = np.ascontiguousarray(np.moveaxis(x, 0, -1))
    x = x.view(dtype=np.complex64)
    x = np.fft.fftshift(x, axes=(3, 4))
    x = np.fft.ifftn(x, axes=(3, 4))
    x = np.stack([x.real, x.imag]).squeeze().astype("float32")
    return x


def invert_ifft(x):
    x = np.ascontiguousarray(np.moveaxis(x, 0, -1))
    x = x.view(dtype=np.complex64)
    x = np.fft.fftn(x, axes=(3, 4))
    x = np.fft.ifftshift(x, axes=(3, 4))
    x = np.stack([x.real, x.imag]).squeeze().astype("float32")
    return x


def normalize(x, file_index):
    shift, scale = 0.0, 1.0
    if cfg.dataset.normalization_scope == "sample":
        if cfg.dataset.normalization == "zscore":
            shift = x.mean((2, 3, 4, 5, 6), keepdims=True)
            scale = x.std((2, 3, 4, 5, 6), keepdims=True)
        if cfg.dataset.normalization == "minmax":
            x_min = x.min((2, 3, 4, 5, 6), keepdims=True)
            x_max = x.max((2, 3, 4, 5, 6), keepdims=True)
            shift = x_min
            scale = x_max - x_min
    if cfg.dataset.normalization_scope == "dataset":
        if cfg.dataset.normalization == "zscore":
            shift = expand_as(data.dataset_stats[file_index]["mean"][None], x)
            scale = expand_as(data.dataset_stats[file_index]["std"][None], x)
        if cfg.dataset.normalization == "minmax":
            x_min = expand_as(data.dataset_stats[file_index]["min"][None], x)
            x_max = expand_as(data.dataset_stats[file_index]["max"][None], x)
            shift = x_min
            scale = x_max - x_min
        scale = torch.tensor(scale, device=x.device)
        shift = torch.tensor(shift, device=x.device)
    return (x - shift) / scale, shift, scale

In [ ]:
data.files

In [78]:
ONESTEP = False
NO_ZONAL = False
# OUT_DIR = f"{CKP}/{'onestep' if ONESTEP else 'autoreg'}"
OUT_DIR = f"/system/user/publicdata/gyrokinetics/dumps/ifft_400M_{'onestep' if ONESTEP else 'autoreg'}"
os.makedirs(OUT_DIR, exist_ok=True)
IDX_0 = 30
IDX_END = 60
STEP = 10
STEP = STEP if ONESTEP else 1
DUMP_STEP = 5

In [7]:
def modify_fds_dat(path):
    with open(path, "r") as infile:
        content = infile.read()
        content = content.replace("DTIM    =  2.000000000000000E-002", "DTIM    =  0.0")
        content = content.replace(
            "NT_REMAIN       =           0", "NT_REMAIN       =           1"
        )
        content = content.replace("TIME    =   192.753733197446     ", "TIME    =   0")

    with open(path, "w") as outfile:
        outfile.write(content)

In [8]:
def modify_input_dat(path):
    with open(path, "r") as infile:
        content = infile.read()
        content = content.replace("READ_FILE  = .false.", "READ_FILE  = .true.")
        content = content.replace("DTIM   = 0.02", "DTIM   = 0.0")
        content = content.replace("out3d_interval = 3", "out3d_interval = 1")
        content = content.replace("keep_dumps = .true.", "! keep_dumps = .true.")
        # content = content.replace("n_procs_s = 4", "n_procs_s = 1")
        # content = content.replace("n_procs_mu = 8", "n_procs_mu = 2")
        # content = content.replace("n_procs_vpar = 8", "n_procs_vpar = 2")

    with open(path, "w") as outfile:
        outfile.write(content)

In [ ]:
losses = []
sample_0 = data[IDX_0]
xt = sample_0.x
if cfg.dataset.spatial_ifft:
    xt = torch.from_numpy(do_ifft(xt.numpy()))
xt = xt.to(device).unsqueeze(0)
itg = sample_0.itg.to(device).unsqueeze(0)
f_idx = sample_0.file_index.item()
timesteps = data.get_timesteps(torch.tensor([0], dtype=torch.long))
files = []
model.eval()
with torch.no_grad():
    for idx in range(IDX_0, IDX_END + 1, STEP):
        if ONESTEP:
            xt = data[idx].x
            if cfg.dataset.spatial_ifft:
                xt = torch.from_numpy(do_ifft(xt.numpy()))
            xt = xt.to(device).unsqueeze(0)
        ts = timesteps[:, idx].to(device)
        xt, shift, scale = normalize(xt, f_idx)
        xt_ = model(xt, timestep=ts, itg=itg)
        if cfg.training.predict_delta:
            xt += xt_
        else:
            xt = xt_
        print("pred", xt.std())
        # denormalize
        xt = xt * scale + shift

        xt = xt.squeeze(0).cpu().numpy()
        if NO_ZONAL and cfg.dataset.spatial_ifft:
            xt = invert_ifft(xt)
            xt[..., 0] = 0.2 * xt[..., 0]
            xt = torch.from_numpy(do_ifft(xt))

        xt = xt.to(device).unsqueeze(0)

        # xt = data[idx].x.to(device).unsqueeze(0)

        b_xt = xt.squeeze(0).cpu().numpy()
        if cfg.dataset.spatial_ifft:
            b_xt = invert_ifft(b_xt)

        b_xt = b_xt.astype("float64").reshape(-1, order="F")
        # dump to file
        if OUT_DIR and (idx % DUMP_STEP) == 0:
            dirtarget = os.path.join(OUT_DIR, f"K{str(int(idx)+1).zfill(2)}")
            os.makedirs(dirtarget, exist_ok=True)
            ftarget = os.path.join(dirtarget, "FDS")
            os.system(
                f"cp {data.files[0].replace("_ifft", "").replace("preprocessed", "raw").replace(".h5", "")}/FDS.dat {dirtarget}"
            )
            os.system(
                f"cp {data.files[0].replace("_ifft", "").replace("preprocessed", "raw").replace(".h5", "")}/input.dat {dirtarget}"
            )
            modify_fds_dat(f"{dirtarget}/FDS.dat")
            modify_input_dat(f"{dirtarget}/input.dat")
            with open(ftarget, "wb") as f:
                files.append(ftarget)
                print(f"Writing file {dirtarget}")
                f.write(b_xt)

            os.system(f"chmod -R 777 {dirtarget}/*")

In [82]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colormaps


def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def distribution_5D(x, **kwargs):
    _ = kwargs
    labels = ["vpar", "vmu", "s", "x", "y"]

    if isinstance(x, torch.Tensor):
        x = x.cpu().detach().numpy()

    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(20, 20))
    for i in range(5):
        for j in range(5):
            if [i, j] not in comb:
                ax[i, j].remove()

    c_map = colormaps["RdBu_r"]

    imin = -1
    for i, j in comb:
        other = tuple([o for o in range(5) if o != i and o != j])
        xx = x[0].mean(other)
        xx[xx == 0] = np.nan
        ax[i, j].matshow(xx, cmap=c_map)

        if i > imin:
            ax[i, j].set_ylabel(labels[i], fontsize=20)
            ax[i, j].set_xlabel(labels[j], fontsize=20)
            imin = i

        force_aspect(ax[i, j])

    return fig

In [ ]:
with open(files[-1], "rb") as fid:
    kt = np.fromfile(fid, dtype=np.float64)
kt = np.reshape(kt, (2, 32, 8, 16, 255, 32), order="F").astype("float32")
distribution_5D(kt)

## Compatibility dump

In [ ]:
IN_DIR = "/system/user/publicdata/gyrokinetics/dumps/onestep_t0/"
OUT_DIR = "/system/user/publicdata/gyrokinetics/dumps/old_onestep_compat/"

for kfile in os.listdir(IN_DIR):
    idx = int(kfile.replace("K", ""))
    dirtarget = os.path.join(OUT_DIR, f"K{str(int(idx)+1).zfill(2)}")
    os.makedirs(dirtarget, exist_ok=True)
    ftarget = os.path.join(dirtarget, "FDS")
    os.system(
        f"cp {data.files[0].replace("_ifft", "").replace("preprocessed", "raw").replace(".h5", "")}/FDS.dat {dirtarget}"
    )
    os.system(
        f"cp {data.files[0].replace("_ifft", "").replace("preprocessed", "raw").replace(".h5", "")}/input.dat {dirtarget}"
    )
    modify_fds_dat(f"{dirtarget}/FDS.dat")
    modify_input_dat(f"{dirtarget}/input.dat")
    os.system(f"rsync -ah --progress {os.path.join(IN_DIR, kfile)} {ftarget}")
    os.system(f"chmod -R 777 {dirtarget}/*")